## Generating Ground Truth Data

In [1]:
# !PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main

# !wget ${PREFIX}/01-agentic-rag/code/ingest.py
# !wget ${PREFIX}/01-agentic-rag/code/rag_helper.py
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/04-evaluation/code/evaluation_utils.py

--2026-06-29 04:54:01--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/04-evaluation/code/evaluation_utils.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3073 (3.0K) [text/plain]
Saving to: ‘evaluation_utils.py.1’

evaluation_utils.py 100%[===================>]   3.00K  --.-KB/s    in 0s      

2026-06-29 04:54:01 (27.3 MB/s) - ‘evaluation_utils.py.1’ saved [3073/3073]



In [2]:
from ingest import load_faq_data
documents = load_faq_data()

In [3]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

85

In [4]:
documents = documents_llm

In [5]:
doc = documents[0]
print(doc["doc_id"])  # changed "doc" to "doc_id"
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [7]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [8]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [9]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [10]:
import json

user_prompt = json.dumps(doc)

In [13]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [14]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [15]:
result = response.output_parsed

print(result)

questions=['I just found this course late — can I still enroll and follow along?', 'Is it okay to join the course after it already started, or am I too late?', 'If I’m joining now, can I still get a certificate somehow?', 'Do late joiners have any restrictions if they want the course certificate?', 'What do I need to do if I want a certificate and I’m starting the course late?']


In [16]:
print(result.questions)

['I just found this course late — can I still enroll and follow along?', 'Is it okay to join the course after it already started, or am I too late?', 'If I’m joining now, can I still get a certificate somehow?', 'Do late joiners have any restrictions if they want the course certificate?', 'What do I need to do if I want a certificate and I’m starting the course late?']


In [17]:
from evaluation_utils import llm_structured

In [19]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['I found the course late — is it still possible to join now?', 'Can I start the course after it’s already begun, or am I too late?', 'If I join the course now, can I still get a certificate?', 'What do I need to do to qualify for the certificate if I’m joining late?', 'Is there still time to submit the project for certification?']


In [20]:
usage.input_tokens, usage.output_tokens

(207, 89)

In [21]:
from evaluation_utils import calc_price

In [22]:
cost = calc_price(usage)

cost

{'input_cost': 0.00015525, 'output_cost': 0.0004005, 'total_cost': 0.00055575}

In [25]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["doc_id"]
    })
    
records

[{'question': 'I found the course late — is it still possible to join now?',
  'document': '74eb249bbf'},
 {'question': 'Can I start the course after it’s already begun, or am I too late?',
  'document': '74eb249bbf'},
 {'question': 'If I join the course now, can I still get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to qualify for the certificate if I’m joining late?',
  'document': '74eb249bbf'},
 {'question': 'Is there still time to submit the project for certification?',
  'document': '74eb249bbf'}]

## Generating Ground Truth for All Documents

In [27]:
# !uv add tqdm pandas

In [28]:
from evaluation_utils import llm_structured_retry

In [31]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["doc_id"]
        })

    return results, usage

In [32]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [34]:
len(ground_truth)

25

In [35]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress


In [36]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/85 [00:00<?, ?it/s]

In [37]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

425

In [38]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost= calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.062076750000000014

In [39]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.062076750000000014

In [40]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [42]:
df_ground_truth.head(10)

,question,document
0,Can I still join the course if I only found it...,74eb249bbf
1,"Is it too late to start the course now, or can...",74eb249bbf
2,"If I join late, will I still be able to get a ...",74eb249bbf
3,What do I need to do to qualify for the certif...,74eb249bbf
4,Are project submissions still open for new stu...,74eb249bbf
5,"I signed up for LLM Zoomcamp, but I still have...",977bf7786c
6,Do I actually need a registration confirmation...,977bf7786c
7,"If I didn’t register for LLM Zoomcamp, can I s...",977bf7786c
8,"Is there some accepted list for this course, o...",977bf7786c
9,What’s the point of the registration form if i...,977bf7786c


In [49]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)

## Search Evaluation

In [50]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [51]:
ground_truth[:5]

[{'question': 'Can I still join the course if I only found it recently?',
  'document': '74eb249bbf'},
 {'question': 'Is it too late to start the course now, or can I still participate?',
  'document': '74eb249bbf'},
 {'question': 'If I join late, will I still be able to get a certificate?',
  'document': '74eb249bbf'},
 {'question': "What do I need to do to qualify for the certificate if I'm joining now?",
  'document': '74eb249bbf'},
 {'question': 'Are project submissions still open for new students who just discovered the course?',
  'document': '74eb249bbf'}]

In [52]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [54]:
documents[:2]

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.",
  'doc_id': '977bf7786c'}]

In [55]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [56]:
q = ground_truth[0]
q

{'question': 'Can I still join the course if I only found it recently?',
 'document': '74eb249bbf'}

In [57]:
doc_id = q["document"]
results = text_search(query=q["question"])

In [60]:
for d in results:
    print(f'{d["doc_id"]} == {doc_id}: {d["doc_id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
9f689c185f == 74eb249bbf: False
69d122f12e == 74eb249bbf: False
977bf7786c == 74eb249bbf: False
04919992b3 == 74eb249bbf: False


In [62]:
relevance = []

for d in results:
    relevance.append(int(d["doc_id"] == doc_id))

relevance

[1, 0, 0, 0, 0]

In [63]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["doc_id"] == doc_id))

    return relevance

In [67]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)

Can I still join the course if I only found it recently?


[1, 0, 0, 0, 0]

In [68]:
q = ground_truth[50]
print(q["question"])
compute_relevance_text(q)

Where do I keep an eye on the LLM Zoomcamp syllabus and deadlines?


[1, 0, 0, 0, 0]

In [71]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [72]:
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

  0%|          | 0/15 [00:00<?, ?it/s]

In [73]:
relevance_total_text

[[1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 1],
 [0, 0, 0, 1, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [75]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["doc_id"] == doc_id))

    return relevance

def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [78]:
relevance_total_sample = compute_relevance_total(ground_truth_sample, text_search)
relevance_total_sample

  0%|          | 0/15 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 1],
 [0, 0, 0, 1, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [77]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/425 [00:00<?, ?it/s]

## Search Evaluation Metrics

In [82]:
# hit rate
cnt = 0

for line in relevance_total_sample:
    if 1 in line:
        cnt += 1

cnt

14

In [83]:
# hit rate for the sample set
cnt / len(relevance_total_sample)

0.9333333333333333

In [84]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt += 1

    return cnt / len(relevance)

In [85]:
hit_rate(relevance_total_sample)

0.9333333333333333

In [86]:
# mean reciprocal rank (mrr)
total_score = 0.0

for line in relevance_total_sample:
    for rank in range(len(line)):
        if line[rank] == 1:
            total_score = total_score + 1 / (rank + 1)
            break
total_score

10.783333333333333

In [88]:
total_score / len(relevance_total_sample)

0.7188888888888889

In [93]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break
    
    return total_score / len(relevance)

In [94]:
mmr(relevance_total_sample)

0.7188888888888889

In [106]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [96]:
evaluate(ground_truth_sample, text_search)

  0%|          | 0/15 [00:00<?, ?it/s]

{'hit_rate': 0.9333333333333333, 'mmr': 0.7188888888888889}

## Search Parameter Tuning

In [97]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [98]:
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

  0%|          | 0/425 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.9176470588235294, 'mmr': 0.8271372549019604}


  0%|          | 0/425 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.9176470588235294, 'mmr': 0.8219999999999996}


  0%|          | 0/425 [00:00<?, ?it/s]

boost=3.0: {'hit_rate': 0.851764705882353, 'mmr': 0.737921568627451}


  0%|          | 0/425 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.8494117647058823, 'mmr': 0.7116862745098037}


  0%|          | 0/425 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.8094117647058824, 'mmr': 0.6852549019607842}


In [107]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [108]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(
                f"Evaluating question_boost={question_boost},"
                f" answer_boost={answer_boost},"
                f" section_boost={section_boost}..."
            )
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/425 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/425 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/425 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/425 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/425 [00:00<?, ?it/s]

In [109]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
18,2.0,4.0,0.1,0.976471,0.894824
34,5.0,10.0,0.2,0.974118,0.894745
3,1.0,2.0,0.1,0.976471,0.894157
35,5.0,10.0,0.5,0.976471,0.894157
19,2.0,4.0,0.2,0.976471,0.894157
33,5.0,10.0,0.1,0.976471,0.893333
4,1.0,2.0,0.2,0.981176,0.892902
7,1.0,4.0,0.2,0.974118,0.890627
20,2.0,4.0,0.5,0.971765,0.888824
6,1.0,4.0,0.1,0.976471,0.888314


In [110]:
def text_search(query):
    boost_dict = {
        "question": 1.0,
        "answer": 2.0,
        "section": 0.1,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )